# ML-Filtered SMA Trading With Anchored WFO

This notebook combines the ML-training and backtesting-framework ideas into one workflow:

1. Train a CUDA cuML classifier on pre-2020 Quant Warehouse optimal-trade `side` labels.
2. Generate fixed 2020+ daily ML predictions.
3. Run anchored walk-forward optimization yearly with `backtesting.py`.
4. For each out-of-sample year, optimize SMA parameters and a portfolio of profitable strategy variants using only prior post-2020 data.
5. Forward test the frozen yearly variant portfolio in the next OOS year.
6. Run Monte Carlo on all out-of-sample trade contribution returns from the combined WFO strategy set.

The ML predictions are injected directly into the in-memory backtest dataframe as an additional SMA filter. During each WFO step, the ML predictions are fixed; only SMA lookbacks and variant weights are optimized in sample.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import itertools
import random
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.optimize import minimize


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
for local_path in (REPO_ROOT, REPO_ROOT.parent / "quant-warehouse"):
    if local_path.exists() and str(local_path) not in sys.path:
        sys.path.insert(0, str(local_path))

warnings.filterwarnings("ignore", category=UserWarning)

from quant_warehouse import Warehouse
from quant_warehouse.feature_engineering import compute_features_worldclass
from quant_warehouse.target_engineering import build_label_panel

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from backtesting import Backtest, Strategy
from quant_orchestrator.pipeline import PipelineContext
from quant_orchestrator.monte_carlo import simulate_return_paths
from quant_orchestrator.platforms.backtesting_frameworks.reporting import build_common_summary, normalize_equity_curve
from quant_orchestrator.platforms.backtesting_frameworks.shared import combine_equity_curves, equal_notional_capital
warnings.filterwarnings("ignore", category=FutureWarning)

Loading BokehJS ...

In [2]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA"]
PROVIDER = "yfinance"
ML_TRAIN_START = "1990-01-01"
ML_TRAIN_END = "2019-12-31"
WFO_START = "2020-01-01"
FIRST_OOS_YEAR = 2021
LAST_OOS_YEAR = 2026
LABEL_K_PARAMS = {"YE": list(range(1, 13))}
ML_FEATURE_COLUMNS = ["open", "high", "low", "close", "volume"]
OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
FAST_WINDOWS = [5, 10, 20, 50]
SLOW_WINDOWS = [60, 100, 150, 200]
CAPITAL_BASE = 100_000.0
MAX_VARIANTS = 8
MAX_VARIANT_WEIGHT = 0.40
MONTE_CARLO_ITERATIONS = 1000
RANDOM_SEED = 20260629

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

WFO_PERIODS = []
for year in range(FIRST_OOS_YEAR, LAST_OOS_YEAR + 1):
    WFO_PERIODS.append(
        {
            "period": str(year),
            "insample_start": WFO_START,
            "insample_end": f"{year - 1}-12-31",
            "oos_start": f"{year}-01-01",
            "oos_end": None if year == LAST_OOS_YEAR else f"{year}-12-31",
        }
    )

config = pd.DataFrame(
    [
        {"setting": "symbols", "value": ", ".join(SYMBOLS)},
        {"setting": "provider", "value": PROVIDER},
        {"setting": "ml_train_window", "value": f"{ML_TRAIN_START} to {ML_TRAIN_END}"},
        {"setting": "wfo_periods", "value": f"{FIRST_OOS_YEAR}-{LAST_OOS_YEAR}, anchored from {WFO_START}"},
        {"setting": "target", "value": f"optimal side labels {LABEL_K_PARAMS}"},
        {"setting": "sma_fast_grid", "value": FAST_WINDOWS},
        {"setting": "sma_slow_grid", "value": SLOW_WINDOWS},
        {"setting": "max_variants", "value": MAX_VARIANTS},
        {"setting": "max_variant_weight", "value": MAX_VARIANT_WEIGHT},
    ]
)
display(config)
display(pd.DataFrame(WFO_PERIODS))

pipeline_context = PipelineContext(metadata={"notebook": "ml_filtered_sma_trading"})
pipeline_context.update({"config": config, "wfo_periods": WFO_PERIODS})

,setting,value
0,symbols,"AAPL, MSFT, NVDA, AMZN, META, GOOGL, TSLA"
1,provider,yfinance
2,ml_train_window,1990-01-01 to 2019-12-31
3,wfo_periods,"2021-2026, anchored from 2020-01-01"
4,target,"optimal side labels {'YE': [1, 2, 3, 4, 5, 6, ..."
5,sma_fast_grid,"[5, 10, 20, 50]"
6,sma_slow_grid,"[60, 100, 150, 200]"
7,max_variants,8
8,max_variant_weight,0.4


,period,insample_start,insample_end,oos_start,oos_end
0,2021,2020-01-01,2020-12-31,2021-01-01,2021-12-31
1,2022,2020-01-01,2021-12-31,2022-01-01,2022-12-31
2,2023,2020-01-01,2022-12-31,2023-01-01,2023-12-31
3,2024,2020-01-01,2023-12-31,2024-01-01,2024-12-31
4,2025,2020-01-01,2024-12-31,2025-01-01,2025-12-31
5,2026,2020-01-01,2025-12-31,2026-01-01,None


PipelineContext(artifacts={'config':               setting                                              value
0             symbols          AAPL, MSFT, NVDA, AMZN, META, GOOGL, TSLA
1            provider                                           yfinance
2     ml_train_window                           1990-01-01 to 2019-12-31
3         wfo_periods                2021-2026, anchored from 2020-01-01
4              target  optimal side labels {'YE': [1, 2, 3, 4, 5, 6, ...
5       sma_fast_grid                                    [5, 10, 20, 50]
6       sma_slow_grid                                [60, 100, 150, 200]
7        max_variants                                                  8
8  max_variant_weight                                                0.4, 'wfo_periods': [{'period': '2021', 'insample_start': '2020-01-01', 'insample_end': '2020-12-31', 'oos_start': '2021-01-01', 'oos_end': '2021-12-31'}, {'period': '2022', 'insample_start': '2020-01-01', 'insample_end': '2021-12-31', '

## Load Quant Warehouse Data, Features, And Labels

Quant Warehouse provides adjusted OHLCV data, technical features, and optimal-trade labels. The model sees only pre-2020 labels. Walk-forward optimization starts after that, so each yearly strategy optimization only sees post-2020 data available before the OOS year.

In [3]:
def normalize_price_frame(prices: pd.DataFrame) -> pd.DataFrame:
    frame = prices.rename(columns=str.lower).copy()
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame = frame.sort_index()
    missing = [column for column in OHLCV_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns: {missing}")
    return frame.loc[:, OHLCV_COLUMNS].astype(float).dropna(subset=["open", "high", "low", "close"])


def load_feature_frame(symbol: str) -> pd.DataFrame:
    prices = Warehouse().read_prices(symbol, provider=PROVIDER, start=ML_TRAIN_START, end=None)
    frame = normalize_price_frame(prices)
    features = compute_features_worldclass(frame.copy())
    features.index = pd.to_datetime(features.index).tz_localize(None)
    return features.sort_index()


def build_symbol_labels(symbol: str, feature_frame: pd.DataFrame) -> pd.DataFrame:
    train_prices = feature_frame.loc[:ML_TRAIN_END, OHLCV_COLUMNS].copy()
    labels = build_label_panel(
        {symbol: train_prices},
        k_params=LABEL_K_PARAMS,
        solver_mode="period_top_k",
        add_rank_labels=True,
        deduplicate=True,
        max_workers=1,
    ).reset_index()
    labels["date"] = pd.to_datetime(labels["date"]).dt.tz_localize(None)
    labels["side"] = labels["side"].astype(str).str.lower()
    labels = labels[labels["side"].isin(["long", "short"])].copy()
    labels["side_code"] = labels["side"].map({"long": 0, "short": 1}).astype("int32")
    labels["symbol"] = symbol
    return labels


feature_frames = {symbol: load_feature_frame(symbol) for symbol in SYMBOLS}
label_frames = {symbol: build_symbol_labels(symbol, feature_frames[symbol]) for symbol in SYMBOLS}

label_summary = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "feature_rows": len(feature_frames[symbol]),
            "label_rows_pre_2020": len(label_frames[symbol]),
            "long_rate": (label_frames[symbol]["side"] == "long").mean(),
            "first_feature_date": feature_frames[symbol].index.min().date(),
            "last_feature_date": feature_frames[symbol].index.max().date(),
        }
        for symbol in SYMBOLS
    ]
)
display(label_summary)
pipeline_context.update({"feature_frames": feature_frames, "label_frames": label_frames, "label_summary": label_summary})

,symbol,feature_rows,label_rows_pre_2020,long_rate,first_feature_date,last_feature_date
0,AAPL,9188,864,0.532407,1990-01-02,2026-06-26
1,MSFT,9188,852,0.535211,1990-01-02,2026-06-26
2,NVDA,6899,602,0.551495,1999-01-22,2026-06-26
3,AMZN,7324,659,0.544765,1997-05-15,2026-06-26
4,META,3546,233,0.562232,2012-05-18,2026-06-26
5,GOOGL,5498,451,0.541020,2004-08-19,2026-06-26
6,TSLA,4023,292,0.513699,2010-06-29,2026-06-26


PipelineContext(artifacts={'config':               setting                                              value
0             symbols          AAPL, MSFT, NVDA, AMZN, META, GOOGL, TSLA
1            provider                                           yfinance
2     ml_train_window                           1990-01-01 to 2019-12-31
3         wfo_periods                2021-2026, anchored from 2020-01-01
4              target  optimal side labels {'YE': [1, 2, 3, 4, 5, 6, ...
5       sma_fast_grid                                    [5, 10, 20, 50]
6       sma_slow_grid                                [60, 100, 150, 200]
7        max_variants                                                  8
8  max_variant_weight                                                0.4, 'wfo_periods': [{'period': '2021', 'insample_start': '2020-01-01', 'insample_end': '2020-12-31', 'oos_start': '2021-01-01', 'oos_end': '2021-12-31'}, {'period': '2022', 'insample_start': '2020-01-01', 'insample_end': '2021-12-31', '

In [4]:
def build_training_table() -> pd.DataFrame:
    rows = []
    for symbol in SYMBOLS:
        labels = label_frames[symbol]
        features = feature_frames[symbol].loc[:, ML_FEATURE_COLUMNS].copy()
        joined = labels.merge(
            features.reset_index().rename(columns={"index": "date"}),
            on="date",
            how="inner",
        )
        joined["symbol"] = symbol
        rows.append(joined)
    table = pd.concat(rows, ignore_index=True).dropna(subset=ML_FEATURE_COLUMNS + ["side_code"])
    return table.sort_values(["date", "symbol"]).reset_index(drop=True)


training_table = build_training_table()
display(training_table[["date", "symbol", "side", "rank_y", *ML_FEATURE_COLUMNS]].head(10))
display(
    pd.DataFrame(
        [
            {
                "training_rows": len(training_table),
                "first_train_label": training_table["date"].min().date(),
                "last_train_label": training_table["date"].max().date(),
                "long_rate": (training_table["side"] == "long").mean(),
            }
        ]
    )
)

,date,symbol,side,rank_y,open,high,low,close,volume
0,1990-01-03,AAPL,short,0.298032,0.265710,0.265710,0.262213,0.262213,207995200.0
1,1990-01-15,MSFT,long,0.986502,0.362143,0.367423,0.360032,0.363727,62467200.0
2,1990-01-29,AAPL,long,0.818866,0.230748,0.234244,0.224629,0.232496,119929600.0
3,1990-01-29,AAPL,short,0.298032,0.230748,0.234244,0.224629,0.232496,119929600.0
4,1990-02-08,AAPL,long,0.737847,0.232496,0.234244,0.225503,0.230748,186636800.0
5,1990-03-26,AAPL,long,0.688657,0.298132,0.304271,0.294625,0.296379,128060800.0
6,1990-04-02,AAPL,long,0.185185,0.280596,0.284980,0.277088,0.282349,148769600.0
7,1990-04-16,AAPL,long,0.737847,0.305148,0.310408,0.303394,0.306901,226889600.0
8,1990-04-16,MSFT,long,0.922535,0.515236,0.521571,0.507845,0.513125,39470400.0
9,1990-04-17,AAPL,short,0.265625,0.303394,0.305148,0.299886,0.303394,131107200.0


,training_rows,first_train_label,last_train_label,long_rate
0,3953,1990-01-03,2019-12-31,0.539337


## Train ML Model And Generate Fixed Predictions

The CUDA cuML classifier is trained once on pre-2020 labels. Its post-2020 predictions are fixed inputs to every WFO period and every SMA variant.

In [5]:
def train_cuml_side_classifier(training_table: pd.DataFrame):
    import cudf
    from cuml.ensemble import RandomForestClassifier

    scaler = StandardScaler()
    x_train_np = scaler.fit_transform(training_table[ML_FEATURE_COLUMNS]).astype("float32")
    y_train_np = training_table["side_code"].astype("int32").to_numpy()

    model = RandomForestClassifier(
        n_estimators=256,
        max_depth=10,
        random_state=RANDOM_SEED,
        n_streams=1,
    )
    started = perf_counter()
    x_gpu = cudf.DataFrame(pd.DataFrame(x_train_np, columns=ML_FEATURE_COLUMNS))
    model.fit(x_gpu, cudf.Series(y_train_np))
    train_seconds = perf_counter() - started
    pred = model.predict(x_gpu).to_numpy().astype(int)
    proba = model.predict_proba(x_gpu).to_numpy()[:, 1]
    metrics = {
        "train_accuracy": accuracy_score(y_train_np, pred),
        "train_macro_f1": f1_score(y_train_np, pred, average="macro", zero_division=0),
        "train_short_roc_auc": roc_auc_score(y_train_np, proba),
        "train_seconds": train_seconds,
    }
    return model, scaler, metrics


ml_model, ml_scaler, train_metrics = train_cuml_side_classifier(training_table)
display(pd.DataFrame([train_metrics]))

[09:42:00] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,train_accuracy,train_macro_f1,train_short_roc_auc,train_seconds
0,0.690362,0.658758,0.792118,1.107651


In [6]:
def predict_symbol_frame(symbol: str) -> pd.DataFrame:
    import cudf

    frame = feature_frames[symbol].loc[WFO_START:, :].copy()
    frame = frame.dropna(subset=ML_FEATURE_COLUMNS).copy()
    x_np = ml_scaler.transform(frame[ML_FEATURE_COLUMNS]).astype("float32")
    x_gpu = cudf.DataFrame(pd.DataFrame(x_np, columns=ML_FEATURE_COLUMNS))
    pred_code = ml_model.predict(x_gpu).to_numpy().astype(int)
    proba_short = ml_model.predict_proba(x_gpu).to_numpy()[:, 1]
    frame["ml_pred_side"] = np.where(pred_code == 0, "long", "short")
    frame["ml_long_filter"] = (pred_code == 0).astype(int)
    frame["ml_short_probability"] = proba_short
    frame["ml_long_probability"] = 1.0 - proba_short
    return frame


prediction_frames = {symbol: predict_symbol_frame(symbol) for symbol in SYMBOLS}
prediction_summary = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "prediction_rows": len(prediction_frames[symbol]),
            "first_prediction_date": prediction_frames[symbol].index.min().date(),
            "last_prediction_date": prediction_frames[symbol].index.max().date(),
            "ml_long_rate": prediction_frames[symbol]["ml_long_filter"].mean(),
        }
        for symbol in SYMBOLS
    ]
)
display(prediction_summary)
display(prediction_frames["AAPL"][["open", "close", "ml_pred_side", "ml_long_probability", "ml_short_probability"]].head(10))

,symbol,prediction_rows,first_prediction_date,last_prediction_date,ml_long_rate
0,AAPL,1629,2020-01-02,2026-06-26,0.510743
1,MSFT,1629,2020-01-02,2026-06-26,0.767342
2,NVDA,1629,2020-01-02,2026-06-26,0.252302
3,AMZN,1629,2020-01-02,2026-06-26,0.559239
4,META,1629,2020-01-02,2026-06-26,0.877225
5,GOOGL,1629,2020-01-02,2026-06-26,0.850829
6,TSLA,1629,2020-01-02,2026-06-26,0.181093


,open,close,ml_pred_side,ml_long_probability,ml_short_probability
date,,,,,
2020-01-02,71.344047,72.333870,long,0.776043,0.223957
2020-01-03,71.563190,71.630623,long,0.635141,0.364859
2020-01-06,70.754014,72.201408,long,0.734879,0.265121
2020-01-07,72.211033,71.861832,long,0.700826,0.299174
2020-01-08,71.565629,73.017845,long,0.776043,0.223957
2020-01-09,73.993195,74.568787,long,0.629933,0.370067
2020-01-10,74.802380,74.737350,long,0.753124,0.246876
2020-01-13,75.052863,76.334084,long,0.770785,0.229215
2020-01-14,76.271471,75.303322,long,0.629933,0.370067


## Backtest Functions

Each run is a MAG7 portfolio with equal notional capital per symbol. `MLFilter` is already in the in-memory dataframe. Setting `use_ml_filter=False` makes the same strategy behave like plain SMA crossover for baseline comparisons.

In [7]:
def build_backtesting_py_frame(
    frame: pd.DataFrame,
    *,
    fast_window: int,
    slow_window: int,
    start: str,
    end: str | None,
    use_ml_filter: bool,
) -> pd.DataFrame:
    fast_col = f"SMA{fast_window}"
    slow_col = f"SMA{slow_window}"
    missing = [column for column in (fast_col, slow_col) if column not in frame.columns]
    if missing:
        raise ValueError(f"Quant Warehouse feature frame is missing columns: {missing}")
    sliced = frame.loc[start:end, :].copy()
    out = sliced.loc[:, OHLCV_COLUMNS + [fast_col, slow_col, "ml_long_filter", "ml_long_probability"]].copy()
    out = out.dropna(subset=[fast_col, slow_col, "ml_long_filter"]).copy()
    out["FastAboveSlow"] = (out[fast_col] > out[slow_col]).astype(int)
    out["MLFilter"] = out["ml_long_filter"].astype(int) if use_ml_filter else 1
    out["Signal"] = ((out["FastAboveSlow"] == 1) & (out["MLFilter"] == 1)).astype(int)
    out = out.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    return out


class MlFilteredSmaStrategy(Strategy):
    trade_fraction = 0.95

    def init(self):
        pass

    def next(self):
        bullish_sma = bool(self.data.FastAboveSlow[-1])
        ml_allows_long = bool(self.data.MLFilter[-1])
        if bullish_sma and ml_allows_long and not self.position:
            self.buy(size=self.trade_fraction)
        elif self.position and (not bullish_sma or not ml_allows_long):
            self.position.close()


def normalize_backtesting_trades(stats: pd.Series, *, symbol: str, variant: str, capital: float) -> pd.DataFrame:
    trades = stats.get("_trades", pd.DataFrame()).copy()
    if trades.empty:
        return pd.DataFrame(columns=["symbol", "variant", "entry_time", "exit_time", "pnl", "return_pct", "capital"])
    trades = trades.rename(columns={"EntryTime": "entry_time", "ExitTime": "exit_time", "PnL": "pnl", "ReturnPct": "return_pct"})
    trades["symbol"] = symbol
    trades["variant"] = variant
    trades["capital"] = capital
    return trades[["symbol", "variant", "entry_time", "exit_time", "pnl", "return_pct", "capital"]]


def run_symbol_backtest(symbol: str, *, fast_window: int, slow_window: int, capital: float, start: str, end: str | None, use_ml_filter: bool = True, variant: str | None = None) -> dict[str, object]:
    frame = build_backtesting_py_frame(
        prediction_frames[symbol],
        fast_window=fast_window,
        slow_window=slow_window,
        start=start,
        end=end,
        use_ml_filter=use_ml_filter,
    )
    started = perf_counter()
    stats = Backtest(frame, MlFilteredSmaStrategy, cash=capital, commission=0.0, trade_on_close=False, exclusive_orders=True).run()
    elapsed = perf_counter() - started
    equity = normalize_equity_curve(stats["_equity_curve"]["Equity"].rename("portfolio_value"))
    trades = normalize_backtesting_trades(stats, symbol=symbol, variant=variant or f"{fast_window}_{slow_window}", capital=capital)
    summary = build_common_summary(framework="backtesting.py", symbol=symbol, equity=equity, elapsed_seconds=elapsed, bars=len(frame), trades=len(trades))
    summary["fast_window"] = fast_window
    summary["slow_window"] = slow_window
    summary["use_ml_filter"] = bool(use_ml_filter)
    summary["ml_long_rate"] = float(frame["MLFilter"].mean())
    summary["sma_long_rate"] = float(frame["FastAboveSlow"].mean())
    summary["combined_signal_rate"] = float(frame["Signal"].mean())
    return {"stats": stats, "summary": summary, "equity": equity, "trades": trades}


def run_portfolio_backtest(*, fast_window: int, slow_window: int, start: str, end: str | None, capital_base: float, use_ml_filter: bool = True) -> dict[str, object]:
    variant = f"{fast_window}_{slow_window}"
    symbol_capital = equal_notional_capital(capital_base, len(SYMBOLS))
    symbol_rows = []
    curves = []
    trade_logs = []
    for symbol in SYMBOLS:
        run = run_symbol_backtest(symbol, fast_window=fast_window, slow_window=slow_window, capital=symbol_capital, start=start, end=end, use_ml_filter=use_ml_filter, variant=variant)
        row = run["summary"].iloc[0].to_dict()
        row["symbol"] = symbol
        symbol_rows.append(row)
        curves.append(run["equity"])
        trade_logs.append(run["trades"])
    portfolio_equity = combine_equity_curves(curves)
    portfolio_summary = build_common_summary(
        framework="backtesting.py",
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=sum(row["elapsed_seconds"] for row in symbol_rows),
        bars=sum(row["bars"] for row in symbol_rows),
        trades=sum(row["trades"] for row in symbol_rows),
    )
    portfolio_summary["fast_window"] = fast_window
    portfolio_summary["slow_window"] = slow_window
    portfolio_summary["use_ml_filter"] = bool(use_ml_filter)
    portfolio_summary["performance_score"] = portfolio_summary["total_return"] + portfolio_summary["max_drawdown"]
    portfolio_summary["variant"] = variant
    return {
        "portfolio_summary": portfolio_summary,
        "symbol_summary": pd.DataFrame(symbol_rows),
        "portfolio_equity": portfolio_equity,
        "trades": pd.concat(trade_logs, ignore_index=True) if trade_logs else pd.DataFrame(),
    }

## Anchored Walk-Forward Optimization

For each OOS year, `backtesting.py` searches the full SMA grid on all prior post-2020 data. We then select profitable variants and optimize long-only variant weights in sample. The selected variants and weights are frozen for that OOS year.

In [8]:
def variant_returns_matrix(runs: dict[tuple[int, int], dict[str, object]], keys: list[tuple[int, int]]) -> pd.DataFrame:
    columns = {}
    for key in keys:
        equity = runs[key]["portfolio_equity"]
        columns[f"{key[0]}_{key[1]}"] = equity.pct_change().dropna()
    return pd.DataFrame(columns).dropna(how="all").fillna(0.0)


def optimize_variant_weights(returns: pd.DataFrame, *, max_weight: float) -> pd.Series:
    if returns.empty:
        raise ValueError("returns matrix is empty")
    n = returns.shape[1]
    if n == 1:
        return pd.Series([1.0], index=returns.columns)
    upper = min(float(max_weight), 1.0)
    if upper * n < 1.0:
        upper = 1.0 / n
    mean = returns.mean().to_numpy(dtype=float)
    cov = returns.cov().to_numpy(dtype=float) + np.eye(n) * 1e-8

    def neg_sharpe(weights: np.ndarray) -> float:
        port_mean = float(weights @ mean)
        port_vol = float(np.sqrt(weights @ cov @ weights))
        return -(port_mean / port_vol) if port_vol > 0 else 1e6

    result = minimize(
        neg_sharpe,
        x0=np.full(n, 1.0 / n),
        method="SLSQP",
        bounds=[(0.0, upper)] * n,
        constraints=[{"type": "eq", "fun": lambda weights: float(weights.sum() - 1.0)}],
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    if not result.success:
        return pd.Series(np.full(n, 1.0 / n), index=returns.columns)
    weights = np.clip(result.x, 0.0, upper)
    weights = weights / weights.sum()
    return pd.Series(weights, index=returns.columns)


def run_insample_search(period: dict[str, str | None]) -> tuple[pd.DataFrame, dict[tuple[int, int], dict[str, object]]]:
    parameter_grid = [(fast, slow) for fast, slow in itertools.product(FAST_WINDOWS, SLOW_WINDOWS) if fast < slow]
    rows = []
    runs = {}
    for fast_window, slow_window in parameter_grid:
        run = run_portfolio_backtest(
            fast_window=fast_window,
            slow_window=slow_window,
            start=str(period["insample_start"]),
            end=str(period["insample_end"]),
            capital_base=CAPITAL_BASE,
            use_ml_filter=True,
        )
        row = run["portfolio_summary"].iloc[0].to_dict()
        rows.append(row)
        runs[(fast_window, slow_window)] = run
    table = pd.DataFrame(rows).sort_values(["performance_score", "sharpe", "total_return"], ascending=[False, False, False]).reset_index(drop=True)
    return table, runs


def select_and_weight_variants(insample_table: pd.DataFrame, insample_runs: dict[tuple[int, int], dict[str, object]]) -> tuple[pd.DataFrame, pd.Series]:
    candidate_table = insample_table[(insample_table["total_return"] > 0) & (insample_table["sharpe"] > 0) & (insample_table["trades"] >= 10)].head(MAX_VARIANTS).copy()
    if candidate_table.empty:
        candidate_table = insample_table.head(1).copy()
    candidate_keys = [(int(row.fast_window), int(row.slow_window)) for row in candidate_table.itertuples(index=False)]
    returns_matrix = variant_returns_matrix(insample_runs, candidate_keys)
    weights = optimize_variant_weights(returns_matrix, max_weight=MAX_VARIANT_WEIGHT)
    weights = weights[weights > 1e-6]
    if weights.empty:
        weights = pd.Series([1.0], index=[f"{candidate_keys[0][0]}_{candidate_keys[0][1]}"])
    weights = weights / weights.sum()
    weights_table = pd.DataFrame({"variant": weights.index, "weight": weights.values}).merge(candidate_table, on="variant", how="left")
    return weights_table, weights

In [9]:
def combine_weighted_variant_equity(weighted_runs: list[dict[str, object]], weights: pd.Series) -> pd.Series:
    combined_index = pd.DatetimeIndex([])
    curves = []
    for run in weighted_runs:
        curve = normalize_equity_curve(run["portfolio_equity"])
        curves.append(curve)
        combined_index = combined_index.union(curve.index)
    combined_index = pd.DatetimeIndex(combined_index).sort_values()
    combined = pd.Series(0.0, index=combined_index, name="portfolio_value")
    for run, curve in zip(weighted_runs, curves, strict=True):
        variant = run["portfolio_summary"]["variant"].iloc[0]
        allocated_capital = CAPITAL_BASE * float(weights[variant])
        aligned = curve.reindex(combined_index).ffill().fillna(curve.iloc[0])
        scaled = aligned / float(aligned.iloc[0]) * allocated_capital
        combined = combined.add(scaled, fill_value=0.0)
    return combined.rename("portfolio_value")


def run_weighted_variant_portfolio(weights: pd.Series, *, start: str, end: str | None, period: str) -> dict[str, object]:
    runs = []
    trades = []
    for variant, weight in weights.items():
        fast_window, slow_window = [int(part) for part in variant.split("_")]
        run = run_portfolio_backtest(fast_window=fast_window, slow_window=slow_window, start=start, end=end, capital_base=CAPITAL_BASE * float(weight), use_ml_filter=True)
        run["portfolio_summary"]["variant"] = variant
        run_trades = run["trades"].copy()
        run_trades["period"] = period
        run_trades["variant_weight"] = float(weight)
        run_trades["portfolio_return_contribution"] = pd.to_numeric(run_trades["pnl"], errors="coerce") / CAPITAL_BASE
        trades.append(run_trades)
        runs.append(run)
    equity = combine_weighted_variant_equity(runs, weights)
    summary = build_common_summary(
        framework="backtesting.py",
        symbol="MAG7_VARIANT_PORTFOLIO",
        equity=equity,
        elapsed_seconds=sum(float(run["portfolio_summary"]["elapsed_seconds"].iloc[0]) for run in runs),
        bars=len(equity),
        trades=sum(int(run["portfolio_summary"]["trades"].iloc[0]) for run in runs),
    )
    summary["period"] = period
    summary["variant_count"] = len(weights)
    summary["performance_score"] = summary["total_return"] + summary["max_drawdown"]
    return {"summary": summary, "equity": equity, "trades": pd.concat(trades, ignore_index=True) if trades else pd.DataFrame(), "runs": runs}


def run_wfo_period(period: dict[str, str | None]) -> dict[str, object]:
    insample_table, insample_runs = run_insample_search(period)
    weights_table, weights = select_and_weight_variants(insample_table, insample_runs)
    weighted_oos = run_weighted_variant_portfolio(weights, start=str(period["oos_start"]), end=period["oos_end"], period=str(period["period"]))
    best_row = insample_table.iloc[0]
    best_oos = run_portfolio_backtest(
        fast_window=int(best_row["fast_window"]),
        slow_window=int(best_row["slow_window"]),
        start=str(period["oos_start"]),
        end=period["oos_end"],
        capital_base=CAPITAL_BASE,
        use_ml_filter=True,
    )
    baseline_oos = run_portfolio_backtest(
        fast_window=int(best_row["fast_window"]),
        slow_window=int(best_row["slow_window"]),
        start=str(period["oos_start"]),
        end=period["oos_end"],
        capital_base=CAPITAL_BASE,
        use_ml_filter=False,
    )
    return {
        "period": period,
        "insample_table": insample_table,
        "weights_table": weights_table,
        "weights": weights,
        "weighted_oos": weighted_oos,
        "best_oos": best_oos,
        "baseline_oos": baseline_oos,
    }


wfo_results = [run_wfo_period(period) for period in WFO_PERIODS]

In [10]:
wfo_summary_rows = []
weights_rows = []
for result in wfo_results:
    period = result["period"]["period"]
    weighted = result["weighted_oos"]["summary"].assign(strategy="optimized_variant_portfolio")
    best = result["best_oos"]["portfolio_summary"].assign(strategy="single_best_variant", period=period)
    baseline = result["baseline_oos"]["portfolio_summary"].assign(strategy="sma_only_baseline", period=period)
    wfo_summary_rows.extend([weighted, best, baseline])
    weights = result["weights_table"].copy()
    weights["period"] = period
    weights_rows.append(weights)

wfo_summary = pd.concat(wfo_summary_rows, ignore_index=True)
wfo_weights = pd.concat(weights_rows, ignore_index=True)

display(Markdown("## Yearly Anchored WFO Out-Of-Sample Summary"))
display(wfo_summary[["period", "strategy", "total_return", "annualized_return", "max_drawdown", "sharpe", "trades", "bars", "performance_score"]])

display(Markdown("## Yearly Selected Variant Weights"))
display(wfo_weights[["period", "variant", "weight", "fast_window", "slow_window", "total_return", "max_drawdown", "sharpe", "trades"]])

## Yearly Anchored WFO Out-Of-Sample Summary

,period,strategy,total_return,annualized_return,max_drawdown,sharpe,trades,bars,performance_score
0,2021,optimized_variant_portfolio,0.2343,0.2343,-0.0609,1.8777,626,252,0.1734
1,2021,single_best_variant,0.2503,0.2503,-0.0607,1.9656,209,1764,0.1896
2,2021,sma_only_baseline,0.4792,0.4792,-0.1282,1.8556,9,1764,0.3510
3,2022,optimized_variant_portfolio,-0.0943,-0.0946,-0.1028,-1.9578,99,251,-0.1971
4,2022,single_best_variant,-0.0758,-0.0761,-0.1056,-1.2585,30,1757,-0.1814
5,2022,sma_only_baseline,-0.1600,-0.1606,-0.1671,-1.1709,7,1757,-0.3271
6,2023,optimized_variant_portfolio,0.1715,0.1730,-0.0736,1.5412,522,250,0.0979
7,2023,single_best_variant,0.1813,0.1829,-0.0745,1.6368,161,1750,0.1068
8,2023,sma_only_baseline,0.5185,0.5236,-0.1190,2.2944,7,1750,0.3995
9,2024,optimized_variant_portfolio,0.2943,0.2943,-0.0720,2.2408,696,252,0.2223


## Yearly Selected Variant Weights

,period,variant,weight,fast_window,slow_window,total_return,max_drawdown,sharpe,trades
0,2021,50_200,0.400000,50,200,0.4601,-0.2608,1.6963,191
1,2021,5_200,0.400000,5,200,0.4172,-0.2500,1.7045,185
2,2021,10_200,0.200000,10,200,0.4205,-0.2584,1.6804,186
3,2022,50_200,0.400000,50,200,0.7700,-0.2608,1.6018,397
4,2022,5_200,0.400000,5,200,0.7050,-0.2500,1.6013,393
5,2022,10_200,0.200000,10,200,0.6990,-0.2584,1.5687,389
6,2023,50_200,0.400000,50,200,0.6447,-0.2608,1.1088,426
7,2023,20_200,0.200000,20,200,0.5425,-0.2564,1.0118,425
8,2023,5_200,0.400000,5,200,0.5323,-0.2500,1.0269,425
9,2024,50_200,0.400000,50,200,0.8960,-0.2608,1.1581,587


## Aggregate OOS Performance And Monte Carlo

The final performance curve stitches together the yearly optimized OOS variant portfolios. The Monte Carlo simulation uses all realized OOS trade contribution returns from those combined yearly strategies.

In [11]:
oos_weighted_curves = []
oos_trade_logs = []
for result in wfo_results:
    curve = result["weighted_oos"]["equity"].copy()
    oos_weighted_curves.append(curve)
    oos_trade_logs.append(result["weighted_oos"]["trades"])

def chain_equity_segments(curves: list[pd.Series], *, capital_base: float) -> pd.Series:
    chained = []
    current_start = float(capital_base)
    for curve in curves:
        normalized = normalize_equity_curve(curve)
        scaled = normalized / float(normalized.iloc[0]) * current_start
        current_start = float(scaled.iloc[-1])
        chained.append(scaled)
    result = pd.concat(chained).sort_index()
    result = result[~result.index.duplicated(keep="last")]
    return result.rename("portfolio_value")

stitched_oos_equity = chain_equity_segments(oos_weighted_curves, capital_base=CAPITAL_BASE)
oos_summary = build_common_summary(
    framework="backtesting.py",
    symbol="MAG7_ANCHORED_WFO",
    equity=stitched_oos_equity,
    elapsed_seconds=sum(float(result["weighted_oos"]["summary"]["elapsed_seconds"].iloc[0]) for result in wfo_results),
    bars=len(stitched_oos_equity),
    trades=sum(int(result["weighted_oos"]["summary"]["trades"].iloc[0]) for result in wfo_results),
)
oos_summary["performance_score"] = oos_summary["total_return"] + oos_summary["max_drawdown"]
display(oos_summary)

all_oos_trades = pd.concat(oos_trade_logs, ignore_index=True)
oos_trade_returns = all_oos_trades["portfolio_return_contribution"].dropna()
mc_result = simulate_return_paths(oos_trade_returns, iterations=MONTE_CARLO_ITERATIONS, horizon=len(oos_trade_returns), seed=RANDOM_SEED, block_size=1)
mc_summary = mc_result.summary.copy()
mc_summary["observed_trade_count"] = len(oos_trade_returns)
mc_summary["observed_trade_return_sum"] = float(oos_trade_returns.sum())
display(mc_summary)

display(Markdown("### OOS Trade Contribution Sample"))
display(all_oos_trades[["period", "symbol", "variant", "variant_weight", "entry_time", "exit_time", "pnl", "portfolio_return_contribution"]].head(10))

,framework,symbol,start,end,bars,trades,initial_equity,final_equity,total_return,annualized_return,max_drawdown,annualized_vol,sharpe,calmar,elapsed_seconds,bars_per_second,performance_score
0,backtesting.py,MAG7_ANCHORED_WFO,2021-01-04,2026-06-26,1376,2690,100000.0,153928.35,0.5393,0.0822,-0.153,0.0976,0.8589,0.5372,1.3409,1026.18,0.3863


,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05,observed_trade_count,observed_trade_return_sum
0,1000,2690,0.676091,0.411209,0.658053,0.986444,-0.041968,-0.063162,2690,0.515513


### OOS Trade Contribution Sample

,period,symbol,variant,variant_weight,entry_time,exit_time,pnl,portfolio_return_contribution
0,2021,AAPL,50_200,0.4,2021-01-06,2021-01-07,26.740128,0.000267
1,2021,AAPL,50_200,0.4,2021-01-08,2021-01-22,157.116306,0.001571
2,2021,AAPL,50_200,0.4,2021-02-02,2021-02-03,1.221605,0.000012
3,2021,AAPL,50_200,0.4,2021-02-04,2021-02-05,51.226252,0.000512
4,2021,AAPL,50_200,0.4,2021-02-18,2021-02-24,-182.396855,-0.001824
5,2021,AAPL,50_200,0.4,2021-02-25,2021-02-26,-91.520839,-0.000915
6,2021,AAPL,50_200,0.4,2021-03-02,2021-03-05,-310.895782,-0.003109
7,2021,AAPL,50_200,0.4,2021-03-10,2021-03-22,-56.907331,-0.000569
8,2021,AAPL,50_200,0.4,2021-03-23,2021-04-20,477.771755,0.004778
9,2021,AAPL,50_200,0.4,2021-04-21,2021-04-22,27.792542,0.000278


In [12]:
readout_lines = [
    "## Notebook Readout",
    f"- Trained one CUDA cuML side classifier on {len(training_table):,} pre-2020 optimal-label rows across MAG7.",
    f"- Generated fixed daily ML predictions for {sum(len(frame) for frame in prediction_frames.values()):,} post-2020 symbol-days.",
    f"- Ran anchored WFO for {len(WFO_PERIODS)} OOS periods and optimized SMA parameters every year using `backtesting.py`.",
    f"- Each year selected up to {MAX_VARIANTS} profitable variants and optimized long-only weights with max weight {MAX_VARIANT_WEIGHT:.0%}.",
]
readout_lines.append(
    f"- Stitched OOS WFO portfolio returned {oos_summary['total_return'].iloc[0]:.2%} with max drawdown {oos_summary['max_drawdown'].iloc[0]:.2%}."
)
readout_lines.append(
    f"- Monte Carlo terminal return p05/p50/p95: {mc_summary['terminal_return_p05'].iloc[0]:.2%} / {mc_summary['terminal_return_p50'].iloc[0]:.2%} / {mc_summary['terminal_return_p95'].iloc[0]:.2%}."
)
readout_lines.append(
    "- The model is fixed before WFO; each yearly loop only optimizes SMA parameters and variant allocation using prior post-2020 data."
)
pipeline_context.update({"training_table": training_table, "ml_metrics": train_metrics, "prediction_frames": prediction_frames, "wfo_results": wfo_results, "wfo_summary": wfo_summary, "wfo_weights": wfo_weights, "oos_summary": oos_summary, "monte_carlo_summary": mc_summary})
display(Markdown("\n".join(readout_lines)))

## Notebook Readout
- Trained one CUDA cuML side classifier on 3,953 pre-2020 optimal-label rows across MAG7.
- Generated fixed daily ML predictions for 11,403 post-2020 symbol-days.
- Ran anchored WFO for 6 OOS periods and optimized SMA parameters every year using `backtesting.py`.
- Each year selected up to 8 profitable variants and optimized long-only weights with max weight 40%.
- Stitched OOS WFO portfolio returned 53.93% with max drawdown -15.30%.
- Monte Carlo terminal return p05/p50/p95: 41.12% / 65.81% / 98.64%.
- The model is fixed before WFO; each yearly loop only optimizes SMA parameters and variant allocation using prior post-2020 data.